 Imports et Connexion Drive

In [ ]:
import os
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.metrics import confusion_matrix, classification_report
from google.colab import drive
from datetime import datetime
import math

drive.mount('/content/drive')

print("Imports chargés et Drive monté")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Imports chargés et Drive monté


# PARAMÈTRES CONFORMES À L'ARTICLE (TABLE 3)

In [ ]:
BATCH_SIZE = 64
EPOCHS = 200
LEARNING_RATE = 0.001
N_TIMESTEPS = 36
N_BANDS = 10

N_STAGES = 3
N_HEAD = 5
KERNEL_SIZE = 3
HIDDEN_DIM = None

OPTIMIZER = 'Adam'

PROCESSED_PATH = Path('/content/drive/MyDrive/Crop_Classification/data/processed')
RAW_DATA_PATH = Path('/content/drive/MyDrive/Crop_Classification/data/raw')
OUTPUTS_PATH = Path('/content/drive/MyDrive/Crop_Classification/outputs')

 DataLoadersModule ALPE

In [ ]:
class ECA(nn.Module):
    """Efficient Channel Attention - Version originale (Wang et al., 2020)"""
    def __init__(self, d_model, gamma=2, b=1):
        super(ECA, self).__init__()
        t = int(abs((math.log(d_model, 2) + b) / gamma))
        kernel_size = t if t % 2 else t + 1
        kernel_size = max(kernel_size, 3)

        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=kernel_size,
                              padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        batch_size, d_model, seq_len = x.shape

        y = self.avg_pool(x)

        y = y.squeeze(-1)
        y = y.unsqueeze(1)

        y = self.conv(y)

        y = y.squeeze(1)
        y = y.unsqueeze(-1)

        y = self.sigmoid(y)

        return x * y


class ALPE(nn.Module):
    def __init__(self, d_model, kernel_size=3):
        super(ALPE, self).__init__()
        self.conv1d = nn.Conv1d(d_model, d_model, kernel_size=kernel_size,
                                padding=kernel_size//2)
        self.eca = ECA(d_model)

    def forward(self, pe, mask):
        pe = pe * mask.unsqueeze(-1)

        x = pe.transpose(1, 2)
        x = self.conv1d(x)

        x = self.eca(x)

        return x.transpose(1, 2)

*CNN* Sub-Module

In [ ]:
class CNNSubModule(nn.Module):
    def __init__(self, d_model, kernel_size=3, dropout_rate=0.4):
        super(CNNSubModule, self).__init__()

        self.conv1 = nn.Conv1d(d_model, d_model, kernel_size, padding=kernel_size//2)
        self.bn1 = nn.BatchNorm1d(d_model)

        self.conv2 = nn.Conv1d(d_model, d_model, kernel_size, padding=kernel_size//2)
        self.bn2 = nn.BatchNorm1d(d_model)

        self.relu = nn.ReLU()

        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        identity = x.transpose(1, 2)

        out = self.conv1(identity)
        out = self.bn1(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = self.relu(out)
        out = out + identity

        out = self.dropout(out)

        return out.transpose(1, 2)

Transformer Sub-Module

In [ ]:
class TransformerSubModule(nn.Module):

    def __init__(self, d_model, nhead, use_alpe=False, seq_len=36, dropout_rate=0.4):
        super(TransformerSubModule, self).__init__()
        self.use_alpe = use_alpe
        self.d_model = d_model
        self.nhead = nhead

        self.dropout1 = nn.Dropout(dropout_rate)
        self.dropout2 = nn.Dropout(dropout_rate)

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

        self.W_o = nn.Linear(d_model, d_model)

        if use_alpe:
            self.alpe = ALPE(d_model)

        self.register_buffer('pe', self._get_sinusoidal_encoding(seq_len, d_model))

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 8),
            nn.ReLU(),
            nn.Linear(d_model * 8, d_model)
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.attn_weights = None

    def _get_sinusoidal_encoding(self, seq_len, d_model):
        pe = torch.zeros(seq_len, d_model)
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                            (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe

    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.shape

        if self.use_alpe and mask is not None:
            pe = self.pe.unsqueeze(0).expand(batch_size, -1, -1)
            pos_encoding = self.alpe(pe, mask)
            x = x + pos_encoding
        else:
            x = x + self.pe.unsqueeze(0)

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        head_dim = d_model // self.nhead
        scale = head_dim ** 0.5

        Q = Q.view(batch_size, seq_len, self.nhead, head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.nhead, head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.nhead, head_dim).transpose(1, 2)

        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / scale

        if mask is not None:
            attn_mask = mask.unsqueeze(1).unsqueeze(1)
            attn_scores = attn_scores.masked_fill(attn_mask == 0, float('-inf'))

        attn_weights = F.softmax(attn_scores, dim=-1)
        self.attn_weights = attn_weights.detach()

        attn_out = torch.matmul(attn_weights, V)

        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        attn_out = self.W_o(attn_out)

        x = self.dropout1(self.norm1(x + attn_out))

        ffn_out = self.ffn(x)

        out = self.dropout2(self.norm2(x + ffn_out))

        return out, attn_weights

CTFusion Module

In [ ]:
class CTFusion(nn.Module):
    def __init__(self, input_dim, hidden_dim, nhead, use_alpe=False, seq_len=36, dropout_rate=0.4):
        super(CTFusion, self).__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        self.proj = nn.Linear(input_dim, hidden_dim) if input_dim != hidden_dim else nn.Identity()

        self.cnn = CNNSubModule(d_model=hidden_dim, kernel_size=3, dropout_rate=dropout_rate)
        self.transformer = TransformerSubModule(d_model=hidden_dim, nhead=nhead, use_alpe=use_alpe, seq_len=seq_len, dropout_rate=dropout_rate)

    def forward(self, x, mask=None):

        x = self.proj(x)

        cnn_out = self.cnn(x)
        trans_out, _ = self.transformer(x, mask)

        combined = torch.cat([cnn_out, trans_out], dim=-1)

        return combined

**MCTNet**

In [ ]:
class MCTNet(nn.Module):
    def __init__(self, input_dim=10, seq_len=36, num_classes=6, nhead=N_HEAD, dropout_rate=0.4):
        super(MCTNet, self).__init__()

        hidden_dim1 = 10
        hidden_dim2 = 20
        hidden_dim3 = 40

        self.stage1 = CTFusion(
            input_dim=input_dim,
            hidden_dim=hidden_dim1,
            nhead=nhead,
            use_alpe=True,
            seq_len=seq_len,
            dropout_rate=dropout_rate
        )
        self.pool1 = nn.MaxPool1d(kernel_size=2, stride=2)

        self.stage2 = CTFusion(
            input_dim=hidden_dim1 * 2,
            hidden_dim=hidden_dim2,
            nhead=nhead,
            use_alpe=False,
            seq_len=seq_len // 2,
            dropout_rate=dropout_rate
        )
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)

        self.stage3 = CTFusion(
            input_dim=hidden_dim2 * 2,
            hidden_dim=hidden_dim3,
            nhead=nhead,
            use_alpe=False,
            seq_len=seq_len // 4,
            dropout_rate=dropout_rate
        )

        self.global_pool = nn.AdaptiveMaxPool1d(1)

        self.classifier = nn.Linear(hidden_dim3 * 2, num_classes)

    def forward(self, x, mask):
        x = self.stage1(x, mask)
        x = x.transpose(1, 2)
        x = self.pool1(x)
        x = x.transpose(1, 2)

        x = self.stage2(x, mask=None)
        x = x.transpose(1, 2)
        x = self.pool2(x)
        x = x.transpose(1, 2)

        x = self.stage3(x, mask=None)

        x = x.transpose(1, 2)
        x = self.global_pool(x)
        x = x.squeeze(-1)

        out = self.classifier(x)
        return out

Fonctions d'entraînement

In [ ]:
def create_mask(x):
    """Crée le masque des données manquantes (Input 2)"""
    return (x.abs().sum(dim=-1) > 0).float()

def train_epoch(model, loader, optimizer, criterion, device):
    """Entraîne le modèle pour une époque"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)
        mask = create_mask(X)

        optimizer.zero_grad()
        outputs = model(X, mask)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += y.size(0)
        correct += predicted.eq(y).sum().item()

    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion, device):
    """Évalue le modèle sur un dataset"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            mask = create_mask(X)
            outputs = model(X, mask)
            loss = criterion(outputs, y)

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += y.size(0)
            correct += predicted.eq(y).sum().item()

    return total_loss / len(loader), correct / total

def calculate_metrics(confusion_matrix):
    """Calcule OA, Kappa et F1"""
    N = confusion_matrix.sum()
    n_classes = confusion_matrix.shape[0]


    sum_diag = np.trace(confusion_matrix)
    oa = sum_diag / N

    row_sums = confusion_matrix.sum(axis=1)
    col_sums = confusion_matrix.sum(axis=0)
    sum_product = (row_sums * col_sums).sum()
    kappa = (N * sum_diag - sum_product) / (N * N - sum_product)

    f1_scores = []
    for i in range(n_classes):
        TP = confusion_matrix[i, i]
        FP = col_sums[i] - TP
        FN = row_sums[i] - TP
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        f1_scores.append(f1)
    macro_f1 = np.mean(f1_scores)

    return {'OA': oa, 'Kappa': kappa, 'F1': macro_f1}

def evaluate_with_confusion_matrix(model, loader, device, n_classes):
    """Évalue et retourne métriques + matrice de confusion"""
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            mask = create_mask(X)
            outputs = model(X, mask)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    cm = confusion_matrix(all_labels, all_preds)
    metrics = calculate_metrics(cm)
    return metrics, cm, all_preds, all_labels

def run_training(model, train_loader, val_loader, test_loader, criterion, optimizer, device,
                 epochs=EPOCHS, state_name="Arkansas"):
    """Boucle d'entraînement complète"""
    best_val_acc = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    print(f"\nDébut de l'entraînement pour {state_name}...")

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f'best_model_{state_name}.pth')

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    model.load_state_dict(torch.load(f'best_model_{state_name}.pth'))
    print(f"\nMeilleure validation accuracy: {best_val_acc:.4f}")

    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    metrics, cm, _, _ = evaluate_with_confusion_matrix(model, test_loader, device, len(np.unique(train_loader.dataset.tensors[1].numpy())))

    print(f"\nRÉSULTATS FINAUX - {state_name.upper()}:")
    print(f"   OA: {metrics['OA']:.4f}")
    print(f"   Kappa: {metrics['Kappa']:.4f}")
    print(f"   F1: {metrics['F1']:.4f}")

    return model, history, metrics, cm

def train_state(model, train_loader, val_loader, test_loader, device, state_name, num_classes, smoothing=0.1):
    """Fonction simplifiée pour entraîner un état"""
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)
    criterion = nn.CrossEntropyLoss(label_smoothing=smoothing)

    model, history, metrics, cm = run_training(
        model=model, train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
        criterion=criterion, optimizer=optimizer,device=device,
        epochs=EPOCHS, state_name=state_name
    )
    return model, history, metrics, cm

Chargement des données

In [ ]:
def load_data_with_correct_split(state_name):
    state_dir = PROCESSED_PATH / state_name.lower()

    train_data = np.load(state_dir / 'train.npz', allow_pickle=True)
    val_data = np.load(state_dir / 'val.npz', allow_pickle=True)
    test_data = np.load(state_dir / 'test.npz', allow_pickle=True)

    X_train, y_train = train_data['X'], train_data['y']
    X_val, y_val = val_data['X'], val_data['y']
    X_test, y_test = test_data['X'], test_data['y']

    with open(state_dir / 'metadata.json', 'r') as f:
        metadata = json.load(f)

    idx_to_code = {int(k): int(v) for k, v in metadata['idx_to_class'].items()}

    if state_name.lower() == 'arkansas':
        MAIN_CODES = {1: 'Corn', 2: 'Cotton', 3: 'Rice', 5: 'Soybeans'}
    else:
        MAIN_CODES = {1: 'Grapes', 2: 'Rice', 3: 'Alfalfa', 4: 'Almonds', 5: 'Pistachios'}

    def convert_to_name(y_array):
        names = []
        for idx in y_array:
            code = idx_to_code.get(int(idx), -1)
            name = MAIN_CODES.get(code, 'Others')
            names.append(name)
        return np.array(names)

    y_train_names = convert_to_name(y_train)
    y_val_names = convert_to_name(y_val)
    y_test_names = convert_to_name(y_test)

    print(f"\n{state_name}:")
    print(f"   - Train: {X_train.shape[0]} samples")
    print(f"   - Val: {X_val.shape[0]} samples")
    print(f"   - Test: {X_test.shape[0]} samples")
    print(f"   - Shape: {X_train.shape[1]} timesteps, {X_train.shape[2]} bands")

    print(f"\n   Distribution RÉELLE (train) :")
    for cls in list(MAIN_CODES.values()) + ['Others']:
        count = np.sum(y_train_names == cls)
        if count > 0:
            print(f"      {cls}: {count}")

    metadata['y_train_names'] = y_train_names
    metadata['y_val_names'] = y_val_names
    metadata['y_test_names'] = y_test_names
    metadata['main_classes'] = list(MAIN_CODES.values()) + ['Others']

    return X_train, y_train, X_val, y_val, X_test, y_test, metadata

ENTRAÎNEMENT ARKANSAS

In [ ]:
print("\n" + "="*60)
print("ENTRAÎNEMENT MCTNet - ARKANSAS")
print("="*60)

X_train, y_train, X_val, y_val, X_test, y_test, metadata = load_data_with_correct_split("arkansas")

y_train_names = metadata['y_train_names']
y_val_names = metadata['y_val_names']
y_test_names = metadata['y_test_names']

print(f"\nDistribution RÉELLE (train) :")
for cls in metadata['main_classes']:
    count = np.sum(y_train_names == cls)
    if count > 0:
        print(f"      {cls}: {count}")

target_classes = ['Soybeans', 'Rice', 'Corn', 'Cotton', 'Others']

train_mask = np.isin(y_train_names, target_classes)
val_mask = np.isin(y_val_names, target_classes)
test_mask = np.isin(y_test_names, target_classes)

X_train_f = X_train[train_mask]
y_train_f = y_train_names[train_mask]

X_val_f = X_val[val_mask]
y_val_f = y_val_names[val_mask]

X_test_f = X_test[test_mask]
y_test_f = y_test_names[test_mask]

print(f"\nAprès filtrage (classes {target_classes}):")
print(f"   Train: {len(X_train_f)} samples")
print(f"   Val: {len(X_val_f)} samples")
print(f"   Test: {len(X_test_f)} samples")

print(f"\nDistribution des classes:")
class_to_idx = {cls: i for i, cls in enumerate(target_classes)}
for cls in target_classes:
    train_count = np.sum(y_train_f == cls)
    val_count = np.sum(y_val_f == cls)
    test_count = np.sum(y_test_f == cls)
    print(f"   {cls}: train={train_count}, val={val_count}, test={test_count}")

y_train_idx = np.array([class_to_idx[name] for name in y_train_f])
y_val_idx = np.array([class_to_idx[name] for name in y_val_f])
y_test_idx = np.array([class_to_idx[name] for name in y_test_f])

num_classes = len(target_classes)
print(f"\nNombre de classes final: {num_classes}")
print(f"   Classes: {target_classes}")

train_ds = TensorDataset(torch.FloatTensor(X_train_f), torch.LongTensor(y_train_idx))
val_ds = TensorDataset(torch.FloatTensor(X_val_f), torch.LongTensor(y_val_idx))
test_ds = TensorDataset(torch.FloatTensor(X_test_f), torch.LongTensor(y_test_idx))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader_ark = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")

model = MCTNet(
    input_dim=N_BANDS,
    seq_len=N_TIMESTEPS,
    num_classes=num_classes,
    nhead=N_HEAD,
    dropout_rate=0.3
).to(device)

print(f"Nombre de paramètres: {sum(p.numel() for p in model.parameters()):,}")

model_arkansas, history_arkansas, metrics_arkansas, cm_arkansas = train_state(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader_ark,
    device=device,
    state_name="Arkansas",
    num_classes=num_classes,
    smoothing=0.1
)

class_names_arkansas = target_classes.copy()


ENTRAÎNEMENT MCTNet - ARKANSAS

arkansas:
   - Train: 1200 samples
   - Val: 300 samples
   - Test: 8500 samples
   - Shape: 36 timesteps, 10 bands

   Distribution RÉELLE (train) :
      Corn: 240
      Cotton: 240
      Rice: 240
      Soybeans: 240
      Others: 240

Distribution RÉELLE (train) :
      Corn: 240
      Cotton: 240
      Rice: 240
      Soybeans: 240
      Others: 240

Après filtrage (classes ['Soybeans', 'Rice', 'Corn', 'Cotton', 'Others']):
   Train: 1200 samples
   Val: 300 samples
   Test: 8500 samples

Distribution des classes:
   Soybeans: train=240, val=60, test=4377
   Rice: train=240, val=60, test=2123
   Corn: train=240, val=60, test=1222
   Cotton: train=240, val=60, test=462
   Others: train=240, val=60, test=316

Nombre de classes final: 5
   Classes: ['Soybeans', 'Rice', 'Corn', 'Cotton', 'Others']

Device: cuda
Nombre de paramètres: 56,718

Début de l'entraînement pour Arkansas...
Epoch  10/200 | Train Loss: 0.7228 | Train Acc: 0.8508 | Val Loss: 0.725

ÉVALUATION & MATRICE DE CONFUSION - ARKANSAS

In [ ]:
print("\n" + "="*60)
print("ÉVALUATION - ARKANSAS")
print("="*60)

print(f"\nRÉSULTATS ARKANSAS:")
print(f"   OA (Overall Accuracy): {metrics_arkansas['OA']:.4f}")
print(f"   Kappa Coefficient: {metrics_arkansas['Kappa']:.4f}")
print(f"   Macro F1 Score: {metrics_arkansas['F1']:.4f}")

print(f"\nMATRICE DE CONFUSION - ARKANSAS:")
print("="*60)

if cm_arkansas.shape[0] < len(class_names_arkansas):
    print(f"La matrice a {cm_arkansas.shape[0]} classes, adaptation des noms...")
    class_names_arkansas = class_names_arkansas[-cm_arkansas.shape[0]:]

print(f"Classes: {class_names_arkansas}")

print(f"\n{'':>20}", end="")
for name in class_names_arkansas:
    short_name = name[:12] if len(name) > 12 else name
    print(f"{short_name:>12}", end="")
print("\n" + "-"*120)

for i, name in enumerate(class_names_arkansas):
    short_name = name[:12] if len(name) > 12 else name
    print(f"{short_name:>20}", end="")
    for j in range(len(class_names_arkansas)):
        print(f"{cm_arkansas[i,j]:>12.0f}", end="")
    print()

print(f"\nMATRICE DE CONFUSION (pourcentages):")
print(f"{'':>20}", end="")
for name in class_names_arkansas:
    short_name = name[:12] if len(name) > 12 else name
    print(f"{short_name:>12}", end="")
print("\n" + "-"*120)

for i, name in enumerate(class_names_arkansas):
    short_name = name[:12] if len(name) > 12 else name
    print(f"{short_name:>20}", end="")
    row_sum = cm_arkansas[i].sum()
    for j in range(len(class_names_arkansas)):
        pct = cm_arkansas[i,j] / row_sum * 100 if row_sum > 0 else 0
        print(f"{pct:>11.1f}%", end="")
    print()

print("\nÉvaluation Arkansas terminée")


ÉVALUATION - ARKANSAS

RÉSULTATS ARKANSAS:
   OA (Overall Accuracy): 0.9618
   Kappa Coefficient: 0.9418
   Macro F1 Score: 0.9284

MATRICE DE CONFUSION - ARKANSAS:
Classes: ['Soybeans', 'Rice', 'Corn', 'Cotton', 'Others']

                        Soybeans        Rice        Corn      Cotton      Others
------------------------------------------------------------------------------------------------------------------------
            Soybeans        4139          18          61          67          92
                Rice          13        2104           1           0           5
                Corn           7           0        1208           0           7
              Cotton          34           1           0         423           4
              Others          12           0           1           2         301

MATRICE DE CONFUSION (pourcentages):
                        Soybeans        Rice        Corn      Cotton      Others
-------------------------------------------------

ENTRAÎNEMENT CALIFORNIA

In [ ]:
print("\n" + "="*60)
print("🏋️ ENTRAÎNEMENT MCTNet - CALIFORNIA")
print("="*60)

X_train, y_train, X_val, y_val, X_test, y_test, metadata = load_data_with_correct_split("california")

idx_to_class = {int(k): int(v) for k, v in metadata['idx_to_class'].items()}

code_to_name = {
    1: 'Grapes',
    2: 'Rice',
    3: 'Alfalfa',
    4: 'Almonds',
    5: 'Pistachios',
    99: 'Others'
}

def convert_to_name(y_array):
    names = []
    for idx in y_array:
        code = idx_to_class.get(int(idx), 99)
        name = code_to_name.get(code, 'Others')
        names.append(name)
    return np.array(names)

y_train_names = convert_to_name(y_train)
y_val_names = convert_to_name(y_val)
y_test_names = convert_to_name(y_test)

print(f"\nDistribution RÉELLE (train) :")
unique, counts = np.unique(y_train_names, return_counts=True)
for name, count in zip(unique, counts):
    print(f"      {name}: {count}")

target_classes = ['Grapes', 'Rice', 'Alfalfa', 'Almonds', 'Pistachios', 'Others']

train_mask = np.isin(y_train_names, target_classes)
val_mask = np.isin(y_val_names, target_classes)
test_mask = np.isin(y_test_names, target_classes)

X_train_f = X_train[train_mask]
y_train_f = y_train_names[train_mask]
X_val_f = X_val[val_mask]
y_val_f = y_val_names[val_mask]
X_test_f = X_test[test_mask]
y_test_f = y_test_names[test_mask]

print(f"\nAprès filtrage (classes {target_classes}):")
print(f"   Train: {len(X_train_f)} samples")
print(f"   Val: {len(X_val_f)} samples")
print(f"   Test: {len(X_test_f)} samples")

print(f"\nDistribution des classes:")
for cls in target_classes:
    train_count = np.sum(y_train_f == cls)
    val_count = np.sum(y_val_f == cls)
    test_count = np.sum(y_test_f == cls)
    print(f"   {cls}: train={train_count}, val={val_count}, test={test_count}")

class_to_idx = {cls: i for i, cls in enumerate(target_classes)}

y_train_idx = np.array([class_to_idx[name] for name in y_train_f])
y_val_idx = np.array([class_to_idx[name] for name in y_val_f])
y_test_idx = np.array([class_to_idx[name] for name in y_test_f])

num_classes = len(target_classes)
print(f"\nNombre de classes final: {num_classes}")
print(f"   Classes: {target_classes}")

train_ds = TensorDataset(torch.FloatTensor(X_train_f), torch.LongTensor(y_train_idx))
val_ds = TensorDataset(torch.FloatTensor(X_val_f), torch.LongTensor(y_val_idx))
test_ds = TensorDataset(torch.FloatTensor(X_test_f), torch.LongTensor(y_test_idx))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader_cal = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = MCTNet(
    input_dim=N_BANDS,
    seq_len=N_TIMESTEPS,
    num_classes=num_classes,
    nhead=N_HEAD,
    dropout_rate=0.15
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Nombre de paramètres: {total_params:,}")

model_california, history_california, metrics_california, cm_california = train_state(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader_cal,
    device=device,
    state_name="California",
    num_classes=num_classes,
    smoothing=0.08
)

class_names_california = target_classes.copy()


🏋️ ENTRAÎNEMENT MCTNet - CALIFORNIA

california:
   - Train: 1440 samples
   - Val: 360 samples
   - Test: 8200 samples
   - Shape: 36 timesteps, 10 bands

   Distribution RÉELLE (train) :
      Grapes: 240
      Rice: 240
      Alfalfa: 240
      Almonds: 240
      Pistachios: 240
      Others: 240

Distribution RÉELLE (train) :
      Alfalfa: 240
      Almonds: 240
      Grapes: 240
      Others: 240
      Pistachios: 240
      Rice: 240

Après filtrage (classes ['Grapes', 'Rice', 'Alfalfa', 'Almonds', 'Pistachios', 'Others']):
   Train: 1440 samples
   Val: 360 samples
   Test: 8200 samples

Distribution des classes:
   Grapes: train=240, val=60, test=1754
   Rice: train=240, val=60, test=1737
   Alfalfa: train=240, val=60, test=674
   Almonds: train=240, val=60, test=483
   Pistachios: train=240, val=60, test=340
   Others: train=240, val=60, test=3212

Nombre de classes final: 6
   Classes: ['Grapes', 'Rice', 'Alfalfa', 'Almonds', 'Pistachios', 'Others']
Device: cuda
Nombre de pa

ÉVALUATION & MATRICE DE CONFUSION - CALIFORNIA

In [ ]:
print("\n" + "="*60)
print("📊 ÉVALUATION - CALIFORNIA")
print("="*60)

print(f"\n📈 RÉSULTATS CALIFORNIA:")
print(f"   OA (Overall Accuracy): {metrics_california['OA']:.4f}")
print(f"   Kappa Coefficient: {metrics_california['Kappa']:.4f}")
print(f"   Macro F1 Score: {metrics_california['F1']:.4f}")

print(f"\n📊 MATRICE DE CONFUSION - CALIFORNIA:")
print("="*60)

class_names_california = target_classes.copy()

if cm_california.shape[0] < len(class_names_california):
    print(f"La matrice a {cm_california.shape[0]} classes, adaptation des noms...")
    class_names_california = class_names_california[-cm_california.shape[0]:]

print(f"\nClasses: {class_names_california}")

print(f"\n{'':>20}", end="")
for name in class_names_california:
    short_name = name[:12] if len(name) > 12 else name
    print(f"{short_name:>12}", end="")
print("\n" + "-"*120)

for i, name in enumerate(class_names_california):
    short_name = name[:12] if len(name) > 12 else name
    print(f"{short_name:>20}", end="")
    for j in range(len(class_names_california)):
        print(f"{cm_california[i,j]:>12.0f}", end="")
    print()

print(f"\nMATRICE DE CONFUSION (pourcentages):")
print(f"{'':>20}", end="")
for name in class_names_california:
    short_name = name[:12] if len(name) > 12 else name
    print(f"{short_name:>12}", end="")
print("\n" + "-"*120)

for i, name in enumerate(class_names_california):
    short_name = name[:12] if len(name) > 12 else name
    print(f"{short_name:>20}", end="")
    row_sum = cm_california[i].sum()
    for j in range(len(class_names_california)):
        pct = cm_california[i,j] / row_sum * 100 if row_sum > 0 else 0
        print(f"{pct:>11.1f}%", end="")
    print()

print("\nÉvaluation California terminée")


📊 ÉVALUATION - CALIFORNIA

📈 RÉSULTATS CALIFORNIA:
   OA (Overall Accuracy): 0.9367
   Kappa Coefficient: 0.9153
   Macro F1 Score: 0.8738

📊 MATRICE DE CONFUSION - CALIFORNIA:

Classes: ['Grapes', 'Rice', 'Alfalfa', 'Almonds', 'Pistachios', 'Others']

                          Grapes        Rice     Alfalfa     Almonds  Pistachios      Others
------------------------------------------------------------------------------------------------------------------------
              Grapes        1508           0          17         166          63           0
                Rice          14        1713           1           7           2           0
             Alfalfa          42           0         557          67           8           0
             Almonds          27           6           7         411          32           0
          Pistachios           7           0           1          52         280           0
              Others           0           0           0           

Résumé final

In [ ]:


print("\n" + "="*60)
print("RÉSUMÉ FINAL DE L'ENTRAÎNEMENT")
print("="*60)

arkansas_classes = ['Soybeans', 'Rice', 'Corn', 'Cotton', 'Others']
california_classes = ['Grapes', 'Rice', 'Alfalfa', 'Almonds', 'Pistachios', 'Others']

print(f"\nARKANSAS:")
print(f"   - OA: {metrics_arkansas['OA']:.4f} ({metrics_arkansas['OA']*100:.2f}%)")
print(f"   - Kappa: {metrics_arkansas['Kappa']:.4f}")
print(f"   - F1: {metrics_arkansas['F1']:.4f}")

print(f"\nCALIFORNIA:")
print(f"   - OA: {metrics_california['OA']:.4f} ({metrics_california['OA']*100:.2f}%)")
print(f"   - Kappa: {metrics_california['Kappa']:.4f}")
print(f"   - F1: {metrics_california['F1']:.4f}")

print("\n" + "="*60)
print("COMPARAISON AVEC L'ARTICLE (Table 4)")
print("="*60)

print(f"\n{'Métrique':<10} {'Article AR':<12} {'Notre AR':<12} {'Article CA':<12} {'Notre CA':<12}")
print("-"*60)
print(f"{'OA':<10} {0.968:<12.4f} {metrics_arkansas['OA']:<12.4f} {0.852:<12.4f} {metrics_california['OA']:<12.4f}")
print(f"{'Kappa':<10} {0.951:<12.4f} {metrics_arkansas['Kappa']:<12.4f} {0.806:<12.4f} {metrics_california['Kappa']:<12.4f}")
print(f"{'F1':<10} {0.933:<12.4f} {metrics_arkansas['F1']:<12.4f} {0.829:<12.4f} {metrics_california['F1']:<12.4f}")

print("\n")


RÉSUMÉ FINAL DE L'ENTRAÎNEMENT

ARKANSAS:
   - OA: 0.9618 (96.18%)
   - Kappa: 0.9418
   - F1: 0.9284

CALIFORNIA:
   - OA: 0.9367 (93.67%)
   - Kappa: 0.9153
   - F1: 0.8738

COMPARAISON AVEC L'ARTICLE (Table 4)

Métrique   Article AR   Notre AR     Article CA   Notre CA    
------------------------------------------------------------
OA         0.9680       0.9618       0.8520       0.9367      
Kappa      0.9510       0.9418       0.8060       0.9153      
F1         0.9330       0.9284       0.8290       0.8738      




# EXPORT DES RÉSULTATS - MATRICES DE CONFUSION, GRAPHIQUES DE COMPARAISON ET RAPPORT D'ENTRAÎNEMENT

In [ ]:
OUTPUTS_PATH.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = OUTPUTS_PATH / f'run_{timestamp}'
run_dir.mkdir(parents=True, exist_ok=True)

print(f" Dossier créé: {run_dir}")

plt.figure(figsize=(10, 8))
sns.heatmap(cm_arkansas, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names_arkansas,
            yticklabels=class_names_arkansas)
plt.title('Confusion Matrix - Arkansas MCTNet')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig(run_dir / 'confusion_matrix_arkansas.png', dpi=150)
plt.close()

plt.figure(figsize=(8, 6))
sns.heatmap(cm_california, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names_california[:len(cm_california)],
            yticklabels=class_names_california[:len(cm_california)])
plt.title('Confusion Matrix - California MCTNet')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig(run_dir / 'confusion_matrix_california.png', dpi=150)
plt.close()

print(" 1. Matrices de confusion sauvegardées")

metrics_names = ['OA', 'Kappa', 'F1']
arkansas_article = [0.968, 0.951, 0.933]
arkansas_ours = [metrics_arkansas['OA'], metrics_arkansas['Kappa'], metrics_arkansas['F1']]
california_article = [0.852, 0.806, 0.829]
california_ours = [metrics_california['OA'], metrics_california['Kappa'], metrics_california['F1']]

x = np.arange(len(metrics_names))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].bar(x - width/2, arkansas_article, width, label='Article', alpha=0.7)
axes[0].bar(x + width/2, arkansas_ours, width, label='Our MCTNet', alpha=0.7)
axes[0].set_xlabel('Metrics')
axes[0].set_ylabel('Score')
axes[0].set_title('Arkansas - Comparison with Article (Table 4)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_names)
axes[0].set_ylim(0, 1)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(x - width/2, california_article, width, label='Article', alpha=0.7)
axes[1].bar(x + width/2, california_ours, width, label='Our MCTNet', alpha=0.7)
axes[1].set_xlabel('Metrics')
axes[1].set_ylabel('Score')
axes[1].set_title('California - Comparison with Article (Table 4)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(metrics_names)
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(run_dir / 'comparison_with_article.png', dpi=150)
plt.close()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history_arkansas['train_loss'], label='Train Loss')
axes[0].plot(history_arkansas['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Arkansas - Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_arkansas['train_acc'], label='Train Acc')
axes[1].plot(history_arkansas['val_acc'], label='Val Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Arkansas - Training & Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(run_dir / 'training_history_arkansas.png', dpi=150)
plt.close()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history_california['train_loss'], label='Train Loss')
axes[0].plot(history_california['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('California - Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_california['train_acc'], label='Train Acc')
axes[1].plot(history_california['val_acc'], label='Val Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('California - Training & Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(run_dir / 'training_history_california.png', dpi=150)
plt.close()

print("\n" + "="*60)
print("🗺️ GÉNÉRATION DES FIGURES 9 & 10 (Style Article - 4 panneaux)")
print("="*60)

print("\n📍 ARKANSAS - Génération Figure style article (4 panneaux)")
print("-"*50)

arkansas_classes = ['Soybeans', 'Rice', 'Corn', 'Cotton', 'Others']

if 'test_loader_ark' in globals():
    ark_results = create_article_style_figure(
        model=model_arkansas,
        test_loader=test_loader_ark,
        class_names=arkansas_classes,
        device=device,
        state_name="Arkansas",
        run_dir=run_dir
    )
else:
    print("❌ test_loader_ark non trouvé")

print("\n📍 CALIFORNIA - Génération Figure style article (4 panneaux)")
print("-"*50)

california_classes = ['Grapes', 'Rice', 'Alfalfa', 'Almonds', 'Pistachios', 'Others']

if 'test_loader_cal' in globals():
    cal_results = create_article_style_figure(
        model=model_california,
        test_loader=test_loader_cal,
        class_names=california_classes,
        device=device,
        state_name="California",
        run_dir=run_dir
    )
else:
    print("❌ test_loader_cal non trouvé")

print("\n" + "="*60)
print("✅ FIGURES STYLE ARTICLE GÉNÉRÉES AVEC SUCCÈS !")
print("="*60)

print("\n📁 TOUS LES RÉSULTATS SONT DANS:", run_dir)

print("\n📋 Fichiers générés:")
for f in sorted(run_dir.iterdir()):
    print("   -", f.name)

📁 Dossier créé: /content/drive/MyDrive/Crop_Classification/outputs/run_20260512_193538
✅ 1. Matrices de confusion sauvegardées
✅ 2. Graphique de comparaison sauvegardé
✅ 3. Historiques d'entraînement sauvegardés

🗺️ GÉNÉRATION DES FIGURES 9 & 10 (Style Article - 4 panneaux)

📍 ARKANSAS - Génération Figure style article (4 panneaux)
--------------------------------------------------
   📊 Collecte des échantillons par classe pour Arkansas...
   📋 Classes disponibles: ['Soybeans', 'Rice', 'Corn', 'Cotton', 'Others']
   🎨 Création de la Zone 1...
   🎨 Création de la Zone 2...


/tmp/ipykernel_16871/2483192787.py:684: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.91, 0.96])


   ✅ Figure sauvegardée: /content/drive/MyDrive/Crop_Classification/outputs/run_20260512_193538/figure_style_article_arkansas.png
   📈 Zone 1 - OA: 0.940 | Zone 2 - OA: 0.943

📍 CALIFORNIA - Génération Figure style article (4 panneaux)
--------------------------------------------------
   📊 Collecte des échantillons par classe pour California...
   📋 Classes disponibles: ['Grapes', 'Rice', 'Alfalfa', 'Almonds', 'Pistachios', 'Others']
   🎨 Création de la Zone 1...
   🎨 Création de la Zone 2...


/tmp/ipykernel_16871/2483192787.py:684: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.91, 0.96])


   ✅ Figure sauvegardée: /content/drive/MyDrive/Crop_Classification/outputs/run_20260512_193538/figure_style_article_california.png
   📈 Zone 1 - OA: 0.854 | Zone 2 - OA: 0.879

✅ FIGURES STYLE ARTICLE GÉNÉRÉES AVEC SUCCÈS !

📋 Structure de chaque figure :
   ┌─────────────────┬─────────────────┐
   │ (a1) MCTNet     │ (a2) Residual   │
   │    Zone 1       │    Zone 1       │
   ├─────────────────┼─────────────────┤
   │ (b1) MCTNet     │ (b2) Residual   │
   │    Zone 2       │    Zone 2       │
   └─────────────────┴─────────────────┘

📁 TOUS LES RÉSULTATS SONT DANS: /content/drive/MyDrive/Crop_Classification/outputs/run_20260512_193538

📋 Fichiers générés:
   - comparison_with_article.png
   - confusion_matrix_arkansas.png
   - confusion_matrix_california.png
   - figure_style_article_arkansas.png
   - figure_style_article_california.png
   - training_history_arkansas.png
   - training_history_california.png
